# SQL Deep Dive — Brazilian E-Commerce (Olist Mock Dataset)

**Portfolio Project 4 — Amazon Data Engineer Internship**

This notebook demonstrates production-quality PostgreSQL across four tiers:

- Tier 1: Basic aggregations, filtering, percentiles
- Tier 2: Multi-table joins, YoY growth, cohort retention
- Tier 3: Window functions, RFM scoring, star-schema joins
- Tier 4: Query optimization — slow vs fast rewrites

Dataset: mock Brazilian E-Commerce (~100K orders, 10K customers, 3K products)

## 0. Setup — Connect to PostgreSQL and load data

In [ ]:
import os
import sys
import time
import textwrap

import pandas as pd
import psycopg2
from psycopg2.extras import RealDictCursor
from IPython.display import display

DB_CONFIG = {
    "host":     os.getenv("DB_HOST",     "localhost"),
    "port":     int(os.getenv("DB_PORT", 5433)),
    "dbname":   os.getenv("DB_NAME",     "ecommerce"),
    "user":     os.getenv("DB_USER",     "postgres"),
    "password": os.getenv("DB_PASSWORD", "postgres"),
}

conn = psycopg2.connect(**DB_CONFIG)
conn.autocommit = True
print("Connected to PostgreSQL:", DB_CONFIG["host"], DB_CONFIG["port"], DB_CONFIG["dbname"])

In [ ]:
def run(sql, params=None, limit=None):
    """Execute SQL and return a DataFrame."""
    start = time.perf_counter()
    with conn.cursor(cursor_factory=RealDictCursor) as cur:
        cur.execute(sql, params)
        rows = cur.fetchall()
    elapsed_ms = (time.perf_counter() - start) * 1000
    df = pd.DataFrame([dict(r) for r in rows])
    print(f"Rows: {len(df):,}   Time: {elapsed_ms:.1f} ms")
    return df if limit is None else df.head(limit)


def explain(sql):
    """Run EXPLAIN ANALYZE and print the plan."""
    with conn.cursor() as cur:
        cur.execute("EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) " + sql)
        rows = cur.fetchall()
    for row in rows:
        print(row[0])

### Load data (run once)

If the tables are empty, run `scripts/load_data.py` from the terminal:

```bash
docker-compose up -d
python scripts/load_data.py
```

The cell below checks row counts and skips loading if data already exists.

In [ ]:
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM orders")
    n = cur.fetchone()[0]

if n == 0:
    scripts_dir = os.path.join(os.path.dirname(os.getcwd()), "scripts")
    load_script = os.path.join(scripts_dir, "load_data.py")
    print(f"No data found. Run:  python {load_script}")
else:
    print(f"Data present: {n:,} orders loaded.")

## 1. Dataset Overview

In [ ]:
tables = ["customers", "sellers", "products", "orders",
          "order_items", "order_reviews", "order_payments"]

counts = {}
with conn.cursor() as cur:
    for t in tables:
        cur.execute(f"SELECT COUNT(*) FROM {t}")
        counts[t] = cur.fetchone()[0]

overview = pd.DataFrame.from_dict(counts, orient="index", columns=["row_count"])
overview.index.name = "table"
display(overview)

In [ ]:
date_range = run("""
    SELECT
        MIN(purchase_timestamp)::DATE AS earliest_order,
        MAX(purchase_timestamp)::DATE AS latest_order,
        COUNT(DISTINCT customer_id)   AS unique_customers,
        COUNT(DISTINCT order_id)      AS total_orders
    FROM orders
""")
display(date_range)

In [ ]:
status_dist = run("""
    SELECT
        status,
        COUNT(*)                                        AS order_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM orders
    GROUP BY status
    ORDER BY order_count DESC
""")
display(status_dist)

## 2. Tier 1 — Basic Queries

### Q1 — What is the total revenue and total number of delivered orders?

In [ ]:
q1 = run("""
    SELECT
        COUNT(DISTINCT o.order_id)          AS total_orders,
        SUM(oi.price + oi.freight_value)    AS total_revenue,
        AVG(oi.price + oi.freight_value)    AS avg_order_revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.status = 'delivered'
""")
display(q1)

### Q2 — What is the average order value broken down by customer state?

In [ ]:
q2 = run("""
    SELECT
        c.state,
        COUNT(DISTINCT o.order_id)                          AS order_count,
        ROUND(AVG(oi.price + oi.freight_value)::NUMERIC, 2) AS avg_order_value,
        ROUND(SUM(oi.price + oi.freight_value)::NUMERIC, 2) AS total_revenue
    FROM customers c
    JOIN orders o       ON o.customer_id = c.customer_id
    JOIN order_items oi ON oi.order_id   = o.order_id
    WHERE o.status = 'delivered'
    GROUP BY c.state
    ORDER BY total_revenue DESC
""")
display(q2)

### Q3 — Which are the top 10 product categories by total revenue?

In [ ]:
q3 = run("""
    SELECT
        p.category,
        COUNT(DISTINCT oi.order_id)                         AS order_count,
        ROUND(SUM(oi.price)::NUMERIC, 2)                    AS product_revenue,
        ROUND(SUM(oi.freight_value)::NUMERIC, 2)            AS freight_revenue,
        ROUND(SUM(oi.price + oi.freight_value)::NUMERIC, 2) AS total_revenue
    FROM products p
    JOIN order_items oi ON oi.product_id = p.product_id
    JOIN orders o       ON o.order_id    = oi.order_id
    WHERE o.status = 'delivered'
    GROUP BY p.category
    ORDER BY total_revenue DESC
    LIMIT 10
""")
display(q3)

### Q4 — How many orders were placed each month?

In [ ]:
q4 = run("""
    SELECT
        DATE_TRUNC('month', purchase_timestamp) AS order_month,
        COUNT(order_id)                         AS order_count,
        COUNT(DISTINCT customer_id)             AS unique_customers
    FROM orders
    GROUP BY DATE_TRUNC('month', purchase_timestamp)
    ORDER BY order_month
""")
display(q4)

### Q5 — Which customers are above the 90th percentile in lifetime spend?

In [ ]:
q5 = run("""
    SELECT
        c.customer_id,
        c.state,
        c.city,
        ROUND(SUM(oi.price + oi.freight_value)::NUMERIC, 2) AS lifetime_spend
    FROM customers c
    JOIN orders o       ON o.customer_id = c.customer_id
    JOIN order_items oi ON oi.order_id   = o.order_id
    WHERE o.status = 'delivered'
    GROUP BY c.customer_id, c.state, c.city
    HAVING SUM(oi.price + oi.freight_value) > (
        SELECT PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY customer_total)
        FROM (
            SELECT SUM(oi2.price + oi2.freight_value) AS customer_total
            FROM orders o2
            JOIN order_items oi2 ON oi2.order_id = o2.order_id
            WHERE o2.status = 'delivered'
            GROUP BY o2.customer_id
        ) t
    )
    ORDER BY lifetime_spend DESC
""", limit=20)
display(q5)

## 3. Tier 2 — Intermediate Queries

### Q6 — Revenue by customer state and product category (4-table join)

In [ ]:
q6_sql = """
    SELECT
        c.state,
        p.category,
        COUNT(DISTINCT o.order_id)                          AS order_count,
        ROUND(SUM(oi.price)::NUMERIC, 2)                    AS product_revenue,
        ROUND(SUM(oi.price + oi.freight_value)::NUMERIC, 2) AS total_revenue
    FROM orders o
    JOIN customers   c  ON c.customer_id = o.customer_id
    JOIN order_items oi ON oi.order_id   = o.order_id
    JOIN products    p  ON p.product_id  = oi.product_id
    WHERE o.status = 'delivered'
    GROUP BY c.state, p.category
    ORDER BY c.state, total_revenue DESC
"""
q6 = run(q6_sql, limit=20)
display(q6)

In [ ]:
print("EXPLAIN ANALYZE — state x category revenue:")
explain(q6_sql.strip())

### Q7 — Monthly revenue with year-over-year growth

In [ ]:
q7_sql = """
    WITH monthly_revenue AS (
        SELECT
            DATE_TRUNC('month', o.purchase_timestamp)       AS order_month,
            EXTRACT(YEAR  FROM o.purchase_timestamp)::INT   AS order_year,
            EXTRACT(MONTH FROM o.purchase_timestamp)::INT   AS order_month_num,
            SUM(oi.price + oi.freight_value)                AS revenue
        FROM orders o
        JOIN order_items oi ON oi.order_id = o.order_id
        WHERE o.status = 'delivered'
        GROUP BY DATE_TRUNC('month', o.purchase_timestamp),
                 EXTRACT(YEAR  FROM o.purchase_timestamp),
                 EXTRACT(MONTH FROM o.purchase_timestamp)
    )
    SELECT
        cur.order_month,
        ROUND(cur.revenue::NUMERIC, 2)  AS revenue,
        ROUND(prev.revenue::NUMERIC, 2) AS prev_year_revenue,
        ROUND(
            ((cur.revenue - prev.revenue) / NULLIF(prev.revenue, 0) * 100)::NUMERIC,
            2
        )                               AS yoy_growth_pct
    FROM monthly_revenue cur
    LEFT JOIN monthly_revenue prev
        ON  prev.order_year      = cur.order_year - 1
        AND prev.order_month_num = cur.order_month_num
    ORDER BY cur.order_month
"""
q7 = run(q7_sql)
display(q7)

### Q8 — Customers who ordered in both 2017 and 2018

In [ ]:
q8_sql = """
    SELECT
        c.customer_id,
        c.state,
        c.city
    FROM customers c
    WHERE c.customer_id IN (
        SELECT customer_id FROM orders
        WHERE EXTRACT(YEAR FROM purchase_timestamp) = 2017
          AND status = 'delivered'
    )
    AND c.customer_id IN (
        SELECT customer_id FROM orders
        WHERE EXTRACT(YEAR FROM purchase_timestamp) = 2018
          AND status = 'delivered'
    )
    ORDER BY c.state, c.customer_id
"""
q8 = run(q8_sql, limit=20)
display(q8)

### Q9 — Daily order count with 7-day rolling average

In [ ]:
q9_sql = """
    WITH daily_orders AS (
        SELECT
            purchase_timestamp::DATE    AS order_date,
            COUNT(order_id)             AS daily_count
        FROM orders
        GROUP BY purchase_timestamp::DATE
    )
    SELECT
        order_date,
        daily_count,
        ROUND(
            AVG(daily_count) OVER (
                ORDER BY order_date
                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
            )::NUMERIC,
            2
        ) AS rolling_7d_avg
    FROM daily_orders
    ORDER BY order_date
"""
q9 = run(q9_sql, limit=30)
display(q9)

### Q10 — Cohort retention: customers returning in subsequent months

In [ ]:
q10_sql = """
    WITH customer_first_order AS (
        SELECT
            customer_id,
            DATE_TRUNC('month', MIN(purchase_timestamp)) AS cohort_month
        FROM orders
        WHERE status = 'delivered'
        GROUP BY customer_id
    ),
    customer_orders AS (
        SELECT
            o.customer_id,
            DATE_TRUNC('month', o.purchase_timestamp) AS order_month
        FROM orders o
        WHERE o.status = 'delivered'
        GROUP BY o.customer_id, DATE_TRUNC('month', o.purchase_timestamp)
    )
    SELECT
        cfo.cohort_month,
        EXTRACT(MONTH FROM AGE(co.order_month, cfo.cohort_month))::INT
                                            AS months_since_first_order,
        COUNT(DISTINCT co.customer_id)      AS retained_customers
    FROM customer_first_order cfo
    JOIN customer_orders co
        ON  co.customer_id = cfo.customer_id
        AND co.order_month >= cfo.cohort_month
    GROUP BY cfo.cohort_month,
             EXTRACT(MONTH FROM AGE(co.order_month, cfo.cohort_month))
    ORDER BY cfo.cohort_month, months_since_first_order
"""
q10 = run(q10_sql, limit=30)
display(q10)

## 4. Tier 3 — Advanced Window Functions

### Q11 — ROW_NUMBER: Top 3 customers per state by lifetime value

`ROW_NUMBER() OVER (PARTITION BY state ORDER BY lifetime_value DESC)` assigns a unique sequential rank within each state partition, resetting to 1 for each new state.

In [ ]:
q11_sql = """
    WITH customer_ltv AS (
        SELECT
            c.customer_id,
            c.state,
            c.city,
            ROUND(SUM(oi.price + oi.freight_value)::NUMERIC, 2) AS lifetime_value
        FROM customers c
        JOIN orders o       ON o.customer_id = c.customer_id
        JOIN order_items oi ON oi.order_id   = o.order_id
        WHERE o.status = 'delivered'
        GROUP BY c.customer_id, c.state, c.city
    ),
    ranked AS (
        SELECT
            customer_id, state, city, lifetime_value,
            ROW_NUMBER() OVER (
                PARTITION BY state
                ORDER BY lifetime_value DESC
            ) AS state_rank
        FROM customer_ltv
    )
    SELECT * FROM ranked
    WHERE state_rank <= 3
    ORDER BY state, state_rank
"""
q11 = run(q11_sql)
display(q11)

### Q12 — RANK + DENSE_RANK: 2nd highest revenue product per category

`RANK` leaves gaps after ties; `DENSE_RANK` does not.  Using `DENSE_RANK = 2` ensures we get the true second tier even when multiple products tie for first.

In [ ]:
q12_sql = """
    WITH product_revenue AS (
        SELECT
            p.product_id,
            p.category,
            ROUND(SUM(oi.price)::NUMERIC, 2) AS total_revenue
        FROM products p
        JOIN order_items oi ON oi.product_id = p.product_id
        JOIN orders o       ON o.order_id    = oi.order_id
        WHERE o.status = 'delivered'
        GROUP BY p.product_id, p.category
    ),
    ranked AS (
        SELECT
            product_id, category, total_revenue,
            RANK()       OVER (PARTITION BY category ORDER BY total_revenue DESC) AS rnk,
            DENSE_RANK() OVER (PARTITION BY category ORDER BY total_revenue DESC) AS dense_rnk
        FROM product_revenue
    )
    SELECT * FROM ranked WHERE dense_rnk = 2
    ORDER BY category, total_revenue DESC
"""
q12 = run(q12_sql)
display(q12)

### Q13 — LAG/LEAD: Day-over-day revenue change percentage

`LAG(col) OVER (ORDER BY date)` fetches the previous row's value in the ordered window — no self-join required.  `LEAD` looks ahead.

In [ ]:
q13_sql = """
    WITH daily_revenue AS (
        SELECT
            purchase_timestamp::DATE        AS order_date,
            SUM(oi.price + oi.freight_value) AS daily_revenue
        FROM orders o
        JOIN order_items oi ON oi.order_id = o.order_id
        WHERE o.status = 'delivered'
        GROUP BY purchase_timestamp::DATE
    )
    SELECT
        order_date,
        ROUND(daily_revenue::NUMERIC, 2)    AS daily_revenue,
        ROUND(
            LAG(daily_revenue) OVER (ORDER BY order_date)::NUMERIC, 2
        )                                   AS prev_day_revenue,
        ROUND(
            (
                (daily_revenue - LAG(daily_revenue) OVER (ORDER BY order_date))
                / NULLIF(LAG(daily_revenue) OVER (ORDER BY order_date), 0)
                * 100
            )::NUMERIC, 2
        )                                   AS dod_change_pct
    FROM daily_revenue
    ORDER BY order_date
"""
q13 = run(q13_sql, limit=30)
display(q13)

### Q14 — SUM OVER ROWS BETWEEN: Running total revenue per customer

`ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` accumulates all rows from the start of the partition through the current row — a classic cumulative sum.

In [ ]:
q14_sql = """
    WITH customer_orders AS (
        SELECT
            o.customer_id,
            o.order_id,
            o.purchase_timestamp,
            SUM(oi.price + oi.freight_value) AS order_value
        FROM orders o
        JOIN order_items oi ON oi.order_id = o.order_id
        WHERE o.status = 'delivered'
        GROUP BY o.customer_id, o.order_id, o.purchase_timestamp
    )
    SELECT
        customer_id,
        order_id,
        purchase_timestamp,
        ROUND(order_value::NUMERIC, 2)  AS order_value,
        ROUND(
            SUM(order_value) OVER (
                PARTITION BY customer_id
                ORDER BY purchase_timestamp
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            )::NUMERIC, 2
        )                               AS running_total
    FROM customer_orders
    ORDER BY customer_id, purchase_timestamp
"""
q14 = run(q14_sql, limit=30)
display(q14)

### Q15 — CTE Pipeline: Customer RFM Segmentation

RFM = Recency (days since last order), Frequency (number of orders), Monetary (total spend).  
`NTILE(5)` divides customers into 5 equal buckets per dimension; the combined score drives the segment label.

In [ ]:
q15_sql = """
    WITH reference_date AS (
        SELECT MAX(purchase_timestamp)::DATE AS max_date
        FROM orders WHERE status = 'delivered'
    ),
    customer_rfm AS (
        SELECT
            o.customer_id,
            (SELECT max_date FROM reference_date)
                - MAX(o.purchase_timestamp)::DATE   AS recency_days,
            COUNT(DISTINCT o.order_id)              AS frequency,
            SUM(oi.price + oi.freight_value)        AS monetary
        FROM orders o
        JOIN order_items oi ON oi.order_id = o.order_id
        WHERE o.status = 'delivered'
        GROUP BY o.customer_id
    ),
    rfm_scores AS (
        SELECT
            customer_id, recency_days, frequency,
            ROUND(monetary::NUMERIC, 2)                     AS monetary,
            NTILE(5) OVER (ORDER BY recency_days ASC)       AS r_score,
            NTILE(5) OVER (ORDER BY frequency    DESC)      AS f_score,
            NTILE(5) OVER (ORDER BY monetary     DESC)      AS m_score
        FROM customer_rfm
    )
    SELECT
        customer_id, recency_days, frequency, monetary,
        r_score, f_score, m_score,
        r_score + f_score + m_score     AS rfm_total,
        CASE
            WHEN r_score >= 4 AND f_score >= 4 THEN 'Champions'
            WHEN r_score >= 3 AND f_score >= 3 THEN 'Loyal Customers'
            WHEN r_score >= 4 AND f_score < 2  THEN 'New Customers'
            WHEN r_score < 2  AND f_score >= 3 THEN 'At Risk'
            WHEN r_score < 2  AND f_score < 2  THEN 'Lost'
            ELSE 'Potential Loyalists'
        END                             AS rfm_segment
    FROM rfm_scores
    ORDER BY rfm_total DESC
"""
q15 = run(q15_sql, limit=30)
display(q15)

segment_counts = run("""
    WITH reference_date AS (
        SELECT MAX(purchase_timestamp)::DATE AS max_date
        FROM orders WHERE status = 'delivered'
    ),
    customer_rfm AS (
        SELECT o.customer_id,
            (SELECT max_date FROM reference_date) - MAX(o.purchase_timestamp)::DATE AS recency_days,
            COUNT(DISTINCT o.order_id) AS frequency,
            SUM(oi.price + oi.freight_value) AS monetary
        FROM orders o JOIN order_items oi ON oi.order_id = o.order_id
        WHERE o.status = 'delivered'
        GROUP BY o.customer_id
    ),
    rfm_scores AS (
        SELECT customer_id,
            NTILE(5) OVER (ORDER BY recency_days ASC)  AS r_score,
            NTILE(5) OVER (ORDER BY frequency    DESC) AS f_score,
            NTILE(5) OVER (ORDER BY monetary     DESC) AS m_score
        FROM customer_rfm
    )
    SELECT
        CASE
            WHEN r_score >= 4 AND f_score >= 4 THEN 'Champions'
            WHEN r_score >= 3 AND f_score >= 3 THEN 'Loyal Customers'
            WHEN r_score >= 4 AND f_score < 2  THEN 'New Customers'
            WHEN r_score < 2  AND f_score >= 3 THEN 'At Risk'
            WHEN r_score < 2  AND f_score < 2  THEN 'Lost'
            ELSE 'Potential Loyalists'
        END                             AS segment,
        COUNT(*)                        AS customer_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM rfm_scores
    GROUP BY 1
    ORDER BY customer_count DESC
""")
print("\nSegment distribution:")
display(segment_counts)

### Q16 — Star schema join: state × category × year revenue + unique customers

In [ ]:
q16_sql = """
    SELECT
        c.state,
        p.category,
        EXTRACT(YEAR FROM o.purchase_timestamp)::INT    AS order_year,
        COUNT(DISTINCT o.customer_id)                   AS unique_customers,
        COUNT(DISTINCT o.order_id)                      AS total_orders,
        ROUND(SUM(oi.price)::NUMERIC, 2)                AS product_revenue,
        ROUND(SUM(oi.price + oi.freight_value)::NUMERIC, 2) AS total_revenue
    FROM orders o
    JOIN customers   c  ON c.customer_id = o.customer_id
    JOIN order_items oi ON oi.order_id   = o.order_id
    JOIN products    p  ON p.product_id  = oi.product_id
    WHERE o.status = 'delivered'
      AND c.state    IN ('SP', 'RJ', 'MG', 'RS', 'PR')
      AND p.category IN ('electronics', 'fashion', 'home_garden', 'sports', 'beauty')
      AND EXTRACT(YEAR FROM o.purchase_timestamp) IN (2017, 2018)
    GROUP BY c.state, p.category, EXTRACT(YEAR FROM o.purchase_timestamp)
    ORDER BY c.state, order_year, total_revenue DESC
"""
q16 = run(q16_sql)
display(q16)

In [ ]:
print("EXPLAIN ANALYZE — star schema join:")
explain(q16_sql.strip())

## 5. Tier 4 — Query Optimization

### Optimization 1 — Sequential scan vs composite index scan

**Root cause:** Without an index on `(customer_id, purchase_timestamp)`, PostgreSQL must read every row of `orders` to filter by one customer and a date range.

**Fix:** `CREATE INDEX idx_orders_customer_date ON orders (customer_id, purchase_timestamp DESC)`

In [ ]:
sample_customer_sql = """
    SELECT customer_id FROM orders
    WHERE status = 'delivered'
    LIMIT 1
"""
with conn.cursor() as cur:
    cur.execute(sample_customer_sql)
    sample_id = cur.fetchone()[0]
print(f"Using customer_id = {sample_id} for demonstration")

In [ ]:
slow_q1 = f"""
    SELECT order_id, purchase_timestamp, status
    FROM orders
    WHERE customer_id = {sample_id}
      AND purchase_timestamp BETWEEN '2017-01-01' AND '2017-12-31'
    ORDER BY purchase_timestamp DESC
"""

print("--- BEFORE (no index, demonstrate with SET enable_indexscan = off) ---")
with conn.cursor() as cur:
    cur.execute("SET enable_indexscan = off; SET enable_bitmapscan = off")
    cur.execute("EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) " + slow_q1)
    for row in cur.fetchall():
        print(row[0])
    cur.execute("RESET enable_indexscan; RESET enable_bitmapscan")

print("\n--- AFTER (index enabled) ---")
explain(slow_q1)

### Optimization 2 — Correlated subquery vs window function

**Root cause:** The correlated subquery re-scans `products` once per row to compute the category average — O(n²) work.

**Fix:** `AVG(price) OVER (PARTITION BY category)` computes all averages in a single pass.

In [ ]:
slow_q2 = """
    SELECT product_id, category, price,
           (SELECT AVG(p2.price) FROM products p2 WHERE p2.category = p.category) AS avg_category_price,
           price - (SELECT AVG(p2.price) FROM products p2 WHERE p2.category = p.category) AS price_vs_avg
    FROM products p
    ORDER BY category, price_vs_avg DESC
"""

fast_q2 = """
    SELECT product_id, category, price,
           ROUND(AVG(price) OVER (PARTITION BY category)::NUMERIC, 2) AS avg_category_price,
           ROUND((price - AVG(price) OVER (PARTITION BY category))::NUMERIC, 2) AS price_vs_avg
    FROM products
    ORDER BY category, price_vs_avg DESC
"""

print("SLOW — correlated subquery:")
explain(slow_q2)
print("\nFAST — window function:")
explain(fast_q2)

In [ ]:
print("Results with window function approach:")
fast_result = run(fast_q2, limit=15)
display(fast_result)

### Optimization 3 — N+1 subqueries vs single aggregating JOIN

**Root cause:** Two correlated subqueries in the SELECT list cause repeated scans of `order_items` per seller row (500 sellers = 1000 subquery executions).

**Fix:** A single `LEFT JOIN` with `GROUP BY` computes all metrics in one pass.

In [ ]:
slow_q3 = """
    SELECT s.seller_id, s.city, s.state,
           (SELECT COUNT(DISTINCT oi.order_id) FROM order_items oi WHERE oi.seller_id = s.seller_id) AS total_orders,
           (SELECT ROUND(SUM(oi.price)::NUMERIC, 2) FROM order_items oi WHERE oi.seller_id = s.seller_id) AS total_revenue
    FROM sellers s
    ORDER BY total_revenue DESC NULLS LAST
"""

fast_q3 = """
    SELECT s.seller_id, s.city, s.state,
           COUNT(DISTINCT oi.order_id)          AS total_orders,
           ROUND(SUM(oi.price)::NUMERIC, 2)     AS total_revenue,
           ROUND(AVG(oi.price)::NUMERIC, 2)     AS avg_item_price
    FROM sellers s
    LEFT JOIN order_items oi ON oi.seller_id = s.seller_id
    GROUP BY s.seller_id, s.city, s.state
    ORDER BY total_revenue DESC NULLS LAST
"""

print("SLOW — N+1 correlated subqueries:")
explain(slow_q3)
print("\nFAST — single aggregating JOIN:")
explain(fast_q3)

In [ ]:
print("Top 10 sellers by revenue:")
seller_perf = run(fast_q3, limit=10)
display(seller_perf)

### Optimization 4 — Missing composite index for state × category × month

**Root cause:** The GROUP BY spans three dimensions across four joined tables.  Without a pre-filter, the planner hashes the full 120K order_items table before joining.

**Fix:** CTE pre-filters orders to a specific year range, shrinking the join input before aggregation.

In [ ]:
slow_q4 = """
    SELECT c.state, p.category,
           DATE_TRUNC('month', o.purchase_timestamp) AS order_month,
           COUNT(DISTINCT o.order_id) AS order_count,
           SUM(oi.price + oi.freight_value) AS total_revenue
    FROM orders o
    JOIN customers   c  ON c.customer_id = o.customer_id
    JOIN order_items oi ON oi.order_id   = o.order_id
    JOIN products    p  ON p.product_id  = oi.product_id
    WHERE o.status = 'delivered'
    GROUP BY c.state, p.category, DATE_TRUNC('month', o.purchase_timestamp)
    ORDER BY c.state, order_month, total_revenue DESC
"""

fast_q4 = """
    WITH delivered_orders AS (
        SELECT order_id, customer_id, purchase_timestamp
        FROM orders
        WHERE status = 'delivered'
          AND purchase_timestamp >= '2017-01-01'
          AND purchase_timestamp <  '2019-01-01'
    ),
    order_revenue AS (
        SELECT
            do_.customer_id,
            DATE_TRUNC('month', do_.purchase_timestamp) AS order_month,
            oi.order_id,
            oi.product_id,
            oi.price + oi.freight_value AS revenue
        FROM delivered_orders do_
        JOIN order_items oi ON oi.order_id = do_.order_id
    )
    SELECT
        c.state,
        p.category,
        or_.order_month,
        COUNT(DISTINCT or_.order_id)                AS order_count,
        ROUND(SUM(or_.revenue)::NUMERIC, 2)         AS total_revenue
    FROM order_revenue or_
    JOIN customers c ON c.customer_id = or_.customer_id
    JOIN products  p ON p.product_id  = or_.product_id
    GROUP BY c.state, p.category, or_.order_month
    ORDER BY c.state, or_.order_month, total_revenue DESC
"""

print("SLOW — full scan, no pre-filter:")
explain(slow_q4)
print("\nFAST — CTE pre-filter:")
explain(fast_q4)

In [ ]:
print("State x category x month revenue (2017-2018):")
q4_result = run(fast_q4, limit=20)
display(q4_result)

## 6. Index Strategy Summary

In [ ]:
indexes = run("""
    SELECT
        schemaname,
        tablename,
        indexname,
        indexdef
    FROM pg_indexes
    WHERE schemaname = 'public'
      AND indexname LIKE 'idx_%'
    ORDER BY tablename, indexname
""")
display(indexes)

In [ ]:
index_usage = run("""
    SELECT
        relname        AS table_name,
        indexrelname   AS index_name,
        idx_scan       AS scans,
        idx_tup_read   AS tuples_read,
        idx_tup_fetch  AS tuples_fetched
    FROM pg_stat_user_indexes
    ORDER BY idx_scan DESC
""")
display(index_usage)

## 7. Key Learnings

1. **Composite indexes eliminate sequential scans** on multi-column filter + sort patterns. The `(customer_id, purchase_timestamp DESC)` index turns a full table scan into an index range scan for customer history queries.

2. **Window functions replace O(n) correlated subqueries.** `AVG(price) OVER (PARTITION BY category)` scans the table once; the equivalent correlated subquery scans once per row.

3. **CTE pre-filters reduce join input size.** Materializing a filtered subset of orders before joining 4 tables significantly cuts the hash aggregate input.

4. **Partial indexes on filtered columns stay small.** `WHERE status != 'delivered'` omits 78% of rows, keeping the index cache-friendly for operational queries.

5. **NTILE-based RFM scoring** produces customer segments in a single query without any application-side logic.

6. **EXPLAIN (ANALYZE, BUFFERS)** is the primary diagnostic tool — actual vs estimated row counts expose stale statistics; buffer hit rates reveal cache effectiveness.

7. **PERCENTILE_CONT** inside a HAVING clause correctly identifies top-spending customers without a second pass over the aggregated result.

8. **LAG/LEAD** compute day-over-day and period-over-period comparisons without self-joins, reducing query complexity and planner overhead.

In [ ]:
conn.close()
print("Connection closed.")